# 05 — Inference API Demo

This notebook demonstrates **Method 2** deployment: wrapping the saved MLflow model
in a FastAPI service and verifying it returns correct predictions.

**Flow:**
```
MLflow (best_model + feature_selector)
    -> FastAPI /predict endpoint (utils/inference_api.py)
    -> HTTP POST with raw DSP features
    -> JSON response: class index + label
```

**Prerequisites:** run `04_main_pipeline.ipynb` at least once so MLflow has a saved run.

In [ ]:
import importlib.util, sys, threading, time, requests
from pathlib import Path
import numpy as np
import mlflow

_BASE_DIR  = Path().resolve()
_UTILS     = _BASE_DIR / 'utils'
MLRUNS_URI = f"file:///{_BASE_DIR / 'mlruns'}"
EXPERIMENT = 'paderborn-bearing-fault'
API_URL    = 'http://127.0.0.1:8000'

print(f'Base dir : {_BASE_DIR}')
print(f'MLruns   : {MLRUNS_URI}')

## Step 1 — Load Model Directly from MLflow

Verify the saved model loads correctly before starting the API server.

In [ ]:
mlflow.set_tracking_uri(MLRUNS_URI)

_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    order_by=['start_time DESC'],
)

if _runs.empty:
    raise RuntimeError('No MLflow runs found. Run 04_main_pipeline.ipynb first.')

_run_id = _runs.iloc[0]['run_id']
print(f"Latest run  : {_run_id[:8]}...")
print(f"Best model  : {_runs.iloc[0].get('params.best_model', '?')}")
print(f"Accuracy    : {_runs.iloc[0].get('metrics.best_accuracy', '?'):.4f}")

selector = mlflow.sklearn.load_model(f'runs:/{_run_id}/feature_selector')
model    = mlflow.sklearn.load_model(f'runs:/{_run_id}/best_model')
print('\nModel and selector loaded successfully.')

## Step 2 — Direct Inference (no API)

Test the model directly in Python first to establish a ground truth,
then compare against the API response in Step 4.

In [ ]:
# Load the feature matrix from the main pipeline cache
import pickle, glob as _glob

_cache_files = sorted(_glob.glob(str(_BASE_DIR / '.feature_cache_*.pkl')))
if not _cache_files:
    raise FileNotFoundError('No feature cache found. Run Section 3f-ii in the main pipeline first.')

with open(_cache_files[-1], 'rb') as f:
    _cache = pickle.load(f)

X_all      = _cache['X']          # (n_signals, 171)
y_all      = _cache['y']
CLASS_NAMES = ['Healthy', 'OR_damage', 'IR_damage']

# Pick 5 random test samples
np.random.seed(42)
_idx     = np.random.choice(len(X_all), size=5, replace=False)
X_sample = X_all[_idx]
y_true   = y_all[_idx]

# Direct prediction
y_direct = model.predict(selector.transform(X_sample))

print('Direct inference results:')
print(f"{'#':<4} {'True':<12} {'Predicted':<12} {'Match'}")
print('-' * 36)
for i, (yt, yp) in enumerate(zip(y_true, y_direct)):
    match = 'OK' if yt == yp else 'WRONG'
    print(f"{i:<4} {CLASS_NAMES[yt]:<12} {CLASS_NAMES[yp]:<12} {match}")

## Step 3 — Start FastAPI Server

Launch `utils/inference_api.py` in a background thread.
The server stays alive for the rest of this notebook session.

In [ ]:
import uvicorn

# Load the FastAPI app from utils/inference_api.py
spec = importlib.util.spec_from_file_location('inference_api', _UTILS / 'inference_api.py')
_api_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(_api_mod)
app = _api_mod.app

# Start uvicorn in a daemon thread so it doesn't block the notebook
_config = uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning')
_server = uvicorn.Server(_config)
_thread = threading.Thread(target=_server.run, daemon=True)
_thread.start()

# Wait for server to be ready
time.sleep(2)
print(f'API server running at {API_URL}')
print(f'Docs available at    {API_URL}/docs')

## Step 4 — Call the API

Send the same 5 samples as HTTP POST requests and compare with direct inference results.

In [ ]:
# Health check
_r = requests.get(f'{API_URL}/health')
print('Health check:', _r.json())

In [ ]:
# Send the 5 samples as a batch POST request
_payload = {'features': X_sample.tolist()}
_r = requests.post(f'{API_URL}/predict', json=_payload)
_r.raise_for_status()
_result = _r.json()

print('API response:')
print(f"  run_id      : {_result['run_id']}")
print(f"  predictions : {_result['predictions']}")
print(f"  labels      : {_result['labels']}")

# Verify API matches direct inference
_api_preds = np.array(_result['predictions'])
assert np.array_equal(_api_preds, y_direct), 'API predictions differ from direct inference!'
print('\nAPI predictions match direct inference OK')

## Step 5 — Interactive Test

Simulate how a factory PLC/SCADA system would call the API with a single new signal.

In [ ]:
# Simulate one new bearing signal (single feature vector)
_single = X_all[0:1]   # first signal in dataset

_r = requests.post(f'{API_URL}/predict', json={'features': _single.tolist()})
_out = _r.json()

print('Single signal prediction:')
print(f"  Class index : {_out['predictions'][0]}")
print(f"  Diagnosis   : {_out['labels'][0]}")

## Production Deployment

To deploy this API on a server (not inside a notebook):

```bash
# Start the server
cd "c:\8. ds"
uvicorn utils.inference_api:app --host 0.0.0.0 --port 8000

# Call from any client (factory PLC, Python script, etc.)
curl -X POST http://<server-ip>:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"features": [[f1, f2, ..., f171]]}'
```

Interactive API docs (Swagger UI): `http://<server-ip>:8000/docs`